# colab_01 — Li 2025: download + per-study QC

Vertical-slice smoke test on the smallest of the three substrate studies: **Li 2025**, GEO GSE237718, temporal cortex, 56 donors, 343k nuclei, purpose-built E2/E3-3/E4 carrier matching. Tests the end-to-end pipeline (download → load → secondary QC → save → audit) on a study that fits cheaply, before scaling to SEA-AD (1.2M) and Haney 2024 (100k).

**Order of operations:**

1. Setup — mount Drive, clone/pull repo, install integration env (section 1)
2. Env capture for this notebook (section 2)
3. Download Li 2025 from GEO into runtime-local cache (section 3)
4. Load into AnnData; print shape + obs/var sanity (section 4)
5. Secondary QC with locked defaults — mito %, filter cells/genes, Scrublet per sample, mito sensitivity sweep (section 5)
6. Audit A partial — niche-critical gene presence in `var_names` (section 6)
7. Save processed AnnData to Drive; cleanup raw if Audit F pressure (section 7)
8. Handoff to `colab_02` (next study) (section 8)

## 1 — Setup

Same pattern as `colab_00`: Drive mount, runtime-local repo clone, `requirements_integration.txt` install.

### 1a — Mount Drive + clone/pull repo + install env

Asserts Python 3.12 before installing — scvi-tools 1.4.x requires it. Adds repo to `sys.path` so `from src.* import ...` resolves.

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info[:2]}"

# Pin numpy first so pip picks numpy-1.x-compatible wheels for pandas, scipy, etc.
# Without this, pip resolves against Colab's pre-installed numpy 2.x, then
# downgrades numpy — leaving a binary ABI mismatch that crashes on import.
# `-q` removed so dependency-conflict warnings stay visible (usually harmless
# Colab-default noise, but worth reading rather than hiding).
!pip install numpy==1.26.4
!pip install -r {REPO_PATH}/requirements_integration.txt

## 2 — Environment capture

Per-notebook env snapshot. Same shape as `colab_00`'s section 3 but condensed into one cell since the pattern is now established.

### 2a — pip freeze + env JSON

Writes `colab_01_<date>_pip_freeze.txt` and `colab_01_<date>_env.json` under `outputs/software_versions/`. Re-captures because Colab runtime resets can produce different resolved versions session to session.

In [ ]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_01"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

env_snapshot = {
    "notebook_id":      NOTEBOOK_ID,
    "date":             TODAY,
    "python_version":   sys.version,
    "platform":         platform.platform(),
    "os_release":       platform.release(),
    "gpu":              _run(["nvidia-smi", "-L"]),
    "nvidia_driver":    _run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]),
    "git_commit":       _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
}
try:
    import torch
    env_snapshot["torch_version"]      = torch.__version__
    env_snapshot["torch_cuda_version"] = torch.version.cuda
except ImportError:
    env_snapshot["torch_version"] = None

ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

## 3 — Download Li 2025

GEO accession **GSE237718**. The series' supplementary directory follows the standard GEO path `https://ftp.ncbi.nlm.nih.gov/geo/series/GSE237nnn/GSE237718/suppl/`. Files there could be (depending on what authors deposited): a single aggregated `.h5ad` / `.rds`, per-sample 10x outputs (`matrix.mtx.gz` + `barcodes.tsv.gz` + `features.tsv.gz`), or `.h5` HDF5 dumps. The notebook fetches everything in `suppl/`, then section 4 branches on what landed.

Raw download lives under runtime-local `/content/li2025_raw/` (not Drive) — it's deleted at end of section 7 per Audit F's keep=False ruling.

### 3a — Fetch GEO supplementary directory

Recursive `wget` of the GEO suppl directory. Time depends on size of deposited matrices; expect several minutes.

In [ ]:
import os, glob, tarfile

GEO_ACCESSION = "GSE237718"
RAW_TAR_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE237nnn/"
    f"{GEO_ACCESSION}/suppl/{GEO_ACCESSION}_RAW.tar"
)
RAW_DIR = "/content/li2025_raw"
os.makedirs(RAW_DIR, exist_ok=True)

# Direct (non-recursive) fetch of the known RAW.tar.
# Recursive wget (-r) first downloads robots.txt and OBEYS it; NCBI disallows
# crawling, so the recursive crawl returns only robots.txt and zero data files.
# A direct file fetch never consults robots.txt, so it just works.
!wget -P {RAW_DIR} {RAW_TAR_URL}

# Layer 1: outer RAW.tar -> 56 per-sample archives (per GEO filelist.txt).
outer = os.path.join(RAW_DIR, f"{GEO_ACCESSION}_RAW.tar")
print(f"Extracting outer tar: {outer}")
with tarfile.open(outer) as t:
    t.extractall(RAW_DIR)
os.remove(outer)  # reclaim ~4.7 GB immediately

# Layer 2: each per-sample .tar.gz -> its 10x files.
# (No-op if the outer tar already held loose files instead of nested archives.)
inner = sorted(glob.glob(f"{RAW_DIR}/**/*.tar.gz", recursive=True))
print(f"Per-sample archives to extract: {len(inner)}")
for arc in inner:
    with tarfile.open(arc) as t:
        t.extractall(os.path.dirname(arc))
    os.remove(arc)

# Discovery: layout + a sample of leaf filenames. This reveals whether per-sample
# 10x files are prefixed (GSMxxxxx_<name>_matrix.mtx.gz) so cell 4a loads correctly.
print("\n=== top-level entries in RAW_DIR ===")
!ls {RAW_DIR} | head -30
print("\n=== sample of leaf files (look for prefixes / per-sample dirs) ===")
!find {RAW_DIR} -type f | head -40

## 4 — Load Li 2025 into AnnData

Branches on file format. Three common GEO deposit shapes are handled:

- **Aggregated h5ad / h5** — single file, load directly with `sc.read_h5ad` / `sc.read_hdf`.
- **Per-sample 10x triples** — directory layout, `sc.read_10x_mtx` per sample, then `anndata.concat`.
- **Single `matrix.mtx.gz` + barcodes + features** — `sc.read_10x_mtx` directly.

If the deposit doesn't match any of these, the load cell will raise and you should inspect the file listing manually.

### 4a — Detect deposit format and load

Sets `adata` (the raw AnnData). Logs which format branch was taken. Does NOT yet attach donor-level APOE metadata — that's 4b.

In [ ]:
import glob, os, shutil, tempfile
import anndata as ad
import scanpy as sc

h5ad_files = sorted(glob.glob(f"{RAW_DIR}/**/*.h5ad", recursive=True))
h5_files   = sorted(glob.glob(f"{RAW_DIR}/**/*.h5",   recursive=True))
# Leading '*' catches GEO per-sample prefixes (GSMxxxxx_<name>_matrix.mtx.gz) as
# well as a bare matrix.mtx.gz. The old exact-name glob missed prefixed deposits.
mtx_files  = sorted(glob.glob(f"{RAW_DIR}/**/*matrix.mtx.gz", recursive=True))

print(f"h5ad: {len(h5ad_files)}  h5: {len(h5_files)}  mtx: {len(mtx_files)}")

def _read_10x_triple(mtx_path):
    """Read one 10x sample from the path to its *matrix.mtx.gz.

    sc.read_10x_mtx requires the canonical trio matrix.mtx.gz / barcodes.tsv.gz /
    features.tsv.gz in one directory. GEO usually ships PREFIXED loose files
    (GSMxxxxx_<name>_matrix.mtx.gz, ...). When prefixed, stage canonical symlinks
    into a temp dir and read from there. Returns (AnnData, sample_id).
    """
    d = os.path.dirname(mtx_path)
    base = os.path.basename(mtx_path)
    prefix = base[: -len("matrix.mtx.gz")]  # "" if already canonical
    barcodes = os.path.join(d, prefix + "barcodes.tsv.gz")
    feats_v3 = os.path.join(d, prefix + "features.tsv.gz")
    feats_v2 = os.path.join(d, prefix + "genes.tsv.gz")   # older 10x naming
    features = feats_v3 if os.path.exists(feats_v3) else feats_v2
    if not (os.path.exists(barcodes) and os.path.exists(features)):
        raise FileNotFoundError(
            f"Incomplete 10x triple for prefix '{prefix}' in {d}: "
            f"barcodes={os.path.exists(barcodes)}, features={os.path.exists(features)}"
        )
    if prefix == "":
        sample_dir, cleanup = d, None
    else:
        sample_dir = tempfile.mkdtemp(prefix="tenx_")
        os.symlink(os.path.abspath(mtx_path),  os.path.join(sample_dir, "matrix.mtx.gz"))
        os.symlink(os.path.abspath(barcodes),  os.path.join(sample_dir, "barcodes.tsv.gz"))
        os.symlink(os.path.abspath(features),  os.path.join(sample_dir, "features.tsv.gz"))
        cleanup = sample_dir
    # default var_names="gene_symbols"; Ensembl IDs preserved in var["gene_ids"].
    a = sc.read_10x_mtx(sample_dir)
    if cleanup:
        shutil.rmtree(cleanup)
    sid = prefix.rstrip("_") if prefix else os.path.basename(d)
    return a, sid

# N_RAW_BARCODES_TOTAL is set only on the multi-sample branch (per-sample prefilter).
# 5b reads it to keep the QC trace's n_cells_initial honest; default None otherwise.
N_RAW_BARCODES_TOTAL = None

if h5ad_files:
    adata = sc.read_h5ad(h5ad_files[0])
    LOAD_BRANCH = f"h5ad ({h5ad_files[0]})"
elif h5_files:
    adata = sc.read_10x_h5(h5_files[0])
    LOAD_BRANCH = f"10x h5 ({h5_files[0]})"
elif len(mtx_files) == 1:
    adata, _ = _read_10x_triple(mtx_files[0])
    LOAD_BRANCH = f"single 10x mtx ({mtx_files[0]})"
elif len(mtx_files) > 1:
    import gc, pandas as pd
    from scipy.sparse import vstack
    # Memory guard: GEO raw 10x deposits often include unfiltered empty-droplet
    # barcodes. Stacking 56 of them simultaneously blew the 52 GB runtime
    # (anndata.concat / vstack both need ~2x the accumulated input alive at once).
    # Apply only the locked min_counts filter to EACH sample BEFORE accumulating:
    # it removes essentially all empty droplets (the memory problem) and is per-cell
    # + independent, so doing it here is result-equivalent to 5b. min_genes and the
    # cross-cell min_cells_per_gene stay in 5b so the QC trace attributes each drop
    # to the right step (the few cells passing counts>=500 but genes<250 are not a
    # memory concern).
    PREFILTER_MIN_COUNTS = 500  # mirrors QC_PARAMS['min_counts'] (defined in 5b, runs later)
    obs_list, X_list, var_ref, n_raw_total = [], [], None, 0
    for m in sorted(mtx_files):
        a, sid = _read_10x_triple(m)
        a.obs["sample_id"] = sid
        n_raw = a.n_obs
        sc.pp.filter_cells(a, min_counts=PREFILTER_MIN_COUNTS)
        print(f"  {sid}: {n_raw} barcodes -> {a.n_obs} cells after min_counts={PREFILTER_MIN_COUNTS} prefilter")
        n_raw_total += n_raw
        if var_ref is None:
            var_ref = a.var.copy()
        elif not a.var_names.equals(var_ref.index):
            raise RuntimeError(
                f"Gene set mismatch in sample '{sid}' — all samples must share the "
                f"same genes (in order) for memory-efficient vstack. If deposits ever "
                f"differ, switch this branch to anndata.concat(join='outer')."
            )
        X_list.append(a.X.tocsr())
        obs_list.append(a.obs.copy())
        del a
        gc.collect()
    adata = ad.AnnData(
        X=vstack(X_list, format="csr"),
        obs=pd.concat(obs_list, ignore_index=True),
        var=var_ref,
    )
    # Free the per-sample accumulators NOW. Without this, X_list keeps a full
    # duplicate of the stacked matrix (~all of X again) alive for the entire rest of
    # the notebook — that dead duplicate is what raised the 5b floor to ~24 GB and
    # pushed the QC copies over the 52 GB runtime ceiling.
    n_samples = len(X_list)
    del X_list, obs_list, var_ref
    gc.collect()
    N_RAW_BARCODES_TOTAL = n_raw_total
    LOAD_BRANCH = (
        f"per-sample 10x prefiltered+stacked ({n_samples} samples, "
        f"{n_raw_total} raw barcodes -> {adata.n_obs} cells)"
    )
else:
    raise RuntimeError(f"Unrecognized GEO deposit shape in {RAW_DIR}; inspect manually.")

adata.var_names_make_unique()
print(f"Load branch: {LOAD_BRANCH}")
print(adata)
fmt = adata.X.getformat() if hasattr(adata.X, "getformat") else type(adata.X).__name__
print(f"X dtype: {adata.X.dtype}  format: {fmt}")
try:
    import psutil
    _m = psutil.virtual_memory()
    print(f"[RAM] after 4a load: {_m.used/1e9:.1f} / {_m.total/1e9:.1f} GB ({_m.percent:.0f}%)")
except Exception:
    pass

### 4b — Attach donor-level metadata (APOE / diagnosis / sex)

Li 2025's GEO supplementary deposit is **raw per-sample 10x only** — no metadata file. Donor-level **APOE genotype**, **pathological diagnosis**, and **sex** live in the GEO **series matrix** (`GSE237718_series_matrix.txt.gz`), keyed by GSM accession. This cell parses it directly from the canonical GEO URL (reproducible; no hand-made CSV) and merges onto `adata.obs` by extracting the `GSM…` id from each cell's `sample_id` (with a sample-title fallback).

**APOE axis (design decision 2026-06-03):** primary stratification is **binary E4-carrier (E3/E4 + E4/E4) vs non-carrier (E3/E3)** — the only axis all three substrate studies can populate (Haney is E3/3 + E4/4 only; E4/E4 is too thin to stand alone). The full 4-level `apoe_genotype` is retained, and E2/E3 is labelled `e2` in `apoe_carrier` and kept in the object but **excluded from the primary eval #2 stratification** (E2 is protective, not a baseline). E2-vs-E3 is reserved as a Li-only optional supplementary probe.

Fail-loud: raises if any `sample_id` fails to map, and prints donor-level cross-tabs that must match Li 2025 (29 AD / 27 control).

In [ ]:
import re, gzip, urllib.request, tempfile, os
import pandas as pd

# Li 2025's GEO suppl tar is raw 10x ONLY — no metadata file. Donor-level APOE
# genotype + diagnosis + sex live in the GEO *series matrix*, keyed by GSM. Parse
# it from the canonical URL (reproducible; no hand-made CSV to trust).
SERIES_MATRIX_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE237nnn/"
    "GSE237718/matrix/GSE237718_series_matrix.txt.gz"
)
_sm_path = os.path.join(tempfile.gettempdir(), "GSE237718_series_matrix.txt.gz")
urllib.request.urlretrieve(SERIES_MATRIX_URL, _sm_path)

# Parse (robust to row order and ragged rows): collect GSM/title/characteristics,
# then assemble per-sample records by column position.
gsm, title, char_rows = [], [], []
with gzip.open(_sm_path, "rt") as fh:
    for line in fh:
        cells = [c.strip().strip('"') for c in line.rstrip("\n").split("\t")]
        tag, vals = cells[0], cells[1:]
        if tag == "!Sample_geo_accession":     gsm = vals
        elif tag == "!Sample_title":           title = vals
        elif tag == "!Sample_characteristics_ch1": char_rows.append(vals)

records = []
for i, g in enumerate(gsm):
    rec = {"gsm": g, "title": title[i] if i < len(title) else None}
    for row in char_rows:
        if i < len(row) and ":" in row[i]:
            k, v = row[i].split(":", 1)
            rec[k.strip()] = v.strip()
    records.append(rec)
meta = pd.DataFrame(records)

def _col(df, *cands):
    for c in cands:
        if c in df.columns:
            return c
    raise KeyError(f"series-matrix characteristics missing {cands}; have {list(df.columns)}")
meta = meta.rename(columns={
    _col(meta, "genotype"): "apoe_genotype",
    _col(meta, "pathological diagnosis", "diagnosis"): "diagnosis",
    _col(meta, "Sex", "sex"): "sex",
})

def _carrier(g):
    g = str(g).upper().replace(" ", "")
    if "E4" in g:
        return "carrier"     # E3/E4 or E4/E4 — primary case group
    if "E2" in g:
        return "e2"          # E2/E3 — protective; kept, but excluded from primary eval #2
    return "noncarrier"      # E3/E3 — baseline
meta["apoe_carrier"] = meta["apoe_genotype"].map(_carrier)

# Map each cell's sample_id -> GSM (stable key in any GEO filename), then attach.
if "sample_id" not in adata.obs.columns:
    raise RuntimeError("adata.obs has no 'sample_id'; loader (4a) did not set it.")

gsm_set = set(meta["gsm"])
title_to_gsm = dict(zip(meta["title"], meta["gsm"]))
def _sid_to_gsm(sid):
    m = re.search(r"GSM\d+", str(sid))
    if m and m.group(0) in gsm_set:
        return m.group(0)
    for t, g in title_to_gsm.items():        # fallback: sample title embedded in sample_id
        if t and t in str(sid):
            return g
    return None

obs_gsm = adata.obs["sample_id"].map(_sid_to_gsm)
unmatched = sorted(adata.obs.loc[obs_gsm.isna(), "sample_id"].astype(str).unique())
if unmatched:
    raise RuntimeError(
        f"{len(unmatched)} sample_id(s) unmatched to series-matrix GSM: "
        f"{unmatched[:10]}{' ...' if len(unmatched) > 10 else ''}"
    )

adata.obs["donor_id"] = obs_gsm.values   # Li 2025: 1 sample == 1 donor
meta_idx = meta.set_index("gsm")
for col in ["apoe_genotype", "apoe_carrier", "diagnosis", "sex"]:
    adata.obs[col] = obs_gsm.map(meta_idx[col]).values

# Fail-loud sanity: donor-level cross-tabs must match Li 2025 (29 AD / 27 control).
donor_tab = adata.obs[["donor_id", "diagnosis", "apoe_genotype", "apoe_carrier"]].drop_duplicates("donor_id")
print(f"donors matched: {donor_tab['donor_id'].nunique()}")
print("\n=== donor counts: diagnosis x apoe_genotype ===")
print(pd.crosstab(donor_tab["diagnosis"], donor_tab["apoe_genotype"]))
print("\n=== donor counts: diagnosis x apoe_carrier (E2 kept separate) ===")
print(pd.crosstab(donor_tab["diagnosis"], donor_tab["apoe_carrier"]))
print("\nobs columns now:", list(adata.obs.columns))

## 5 — Secondary QC (locked defaults)

Per the locked spec (2026-05-21) and `src/qc.py`:

- min 500 counts / nucleus
- min 250 genes / nucleus
- max 5% mito (tight because **nuclei** — high mito = cytoplasmic contamination)
- min 10 cells / gene
- Scrublet per sample
- Mito threshold sensitivity check at 3% / 5% / 10%

Two-layer principle: Li 2025's published per-study QC + cell-type labels are inherited (if shipped on GEO); this is the **secondary** uniform filter on top.

Implemented inline in scanpy; will be lifted into `src/qc.py:secondary_qc` after the pattern stabilizes across all three studies.

### 5a — Annotate mito % and QC metrics

Detects var_name format (HGNC `MT-*` vs Ensembl `ENSG...`) and computes `pct_counts_mt` accordingly. Logs n_cells, n_genes, mito gene count for sanity.

In [ ]:
looks_ensembl = adata.var_names.str.startswith("ENSG").mean() > 0.5
if looks_ensembl:
    # TODO — Li 2025 var_names appear to be Ensembl IDs. Need a gene_symbols column
    # on adata.var (often shipped by GEO as 'gene_symbols' or 'feature_name') to
    # detect mito genes via 'MT-' prefix. Set the column name here:
    SYMBOL_COL = None  # e.g. "gene_symbols"
    if SYMBOL_COL and SYMBOL_COL in adata.var.columns:
        adata.var["mt"] = adata.var[SYMBOL_COL].astype(str).str.startswith("MT-")
    else:
        raise RuntimeError(
            "var_names are Ensembl but no gene_symbols column identified. "
            "Inspect adata.var.columns and set SYMBOL_COL."
        )
else:
    adata.var["mt"] = adata.var_names.str.startswith("MT-")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, percent_top=None, log1p=False)
if int(adata.var["mt"].sum()) == 0:
    raise RuntimeError(
        "Zero mito genes detected. If using SYMBOL_COL, verify it contains HGNC "
        "names with 'MT-' prefix, not Ensembl IDs."
    )

print(f"shape:        {adata.shape}")
print(f"var format:   {'Ensembl' if looks_ensembl else 'HGNC symbols'}")
print(f"mito genes:   {int(adata.var['mt'].sum())}")
print(adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe())

### 5b — Apply locked thresholds

Filters in order: cells by counts, cells by gene count, cells by mito %, then genes by min cell count. Records before/after counts for the audit report.

Before the (destructive) mito filter, snapshots the per-cell distribution into `pre_mito_obs` — cell 5d's sensitivity sweep needs the high-mito cells this step removes.

In [ ]:
import gc
try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:20s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except Exception:
    def _ram(tag):
        print(f"[RAM] {tag}: psutil unavailable")

QC_PARAMS = {
    "min_counts":         500,
    "min_genes":          250,
    "max_mito_pct":       5.0,
    "min_cells_per_gene": 10,
}

# True raw barcode total (see 4a). On the multi-sample branch 4a already applied the
# min_counts prefilter per sample, so adata.n_obs here is already post-prefilter;
# N_RAW_BARCODES_TOTAL carries the honest pre-filter denominator for the QC funnel.
n_cells_0 = N_RAW_BARCODES_TOTAL if N_RAW_BARCODES_TOTAL is not None else adata.n_obs
n_genes_0 = adata.n_vars

# --- Single-copy cell filter -----------------------------------------------------
# 5a already computed total_counts / n_genes_by_counts / pct_counts_mt, so the three
# locked cell thresholds are pure boolean masks on obs — no matrix touch. The old
# cell did filter_cells -> filter_cells -> boolean mito subset (three sequential
# .copy()s, each transiently doubling X). Combining them into ONE mask + ONE copy is
# what keeps 5b under the 52 GB ceiling. Cumulative mask sums reproduce the exact
# funnel numbers the sequential version reported (filters are per-cell + cumulative).
_ram("5b start")
counts_mask = (adata.obs["total_counts"]      >= QC_PARAMS["min_counts"]).to_numpy()
genes_mask  = (adata.obs["n_genes_by_counts"] >= QC_PARAMS["min_genes"]).to_numpy()
mito_mask   = (adata.obs["pct_counts_mt"]     <  QC_PARAMS["max_mito_pct"]).to_numpy()

n_after_counts = int(counts_mask.sum())
n_after_genes  = int((counts_mask & genes_mask).sum())
cell_keep      = counts_mask & genes_mask & mito_mask
n_after_mito   = int(cell_keep.sum())

# pre_mito_obs: post count/gene filter, PRE mito filter — the pool 5d's sensitivity
# sweep needs (the mito filter is destructive to the high-mito cells it tests). obs
# only (one row per cell, no matrix), so this is cheap.
pre_mito_obs = adata.obs[counts_mask & genes_mask].copy()

adata = adata[cell_keep].copy()
gc.collect()
_ram("after cell filter")

# Gene filter on the surviving cells — one more copy, unavoidable (var-axis op).
sc.pp.filter_genes(adata, min_cells=QC_PARAMS["min_cells_per_gene"])
gc.collect()
_ram("after gene filter")

qc_trace = {
    "params": QC_PARAMS,
    "n_cells_initial":          n_cells_0,
    "prefiltered_in_4a":        N_RAW_BARCODES_TOTAL is not None,
    "n_cells_after_min_counts": n_after_counts,
    "n_cells_after_min_genes":  n_after_genes,
    "n_cells_after_max_mito":   n_after_mito,
    "n_cells_final":            adata.n_obs,
    "n_genes_initial":          n_genes_0,
    "n_genes_final":            adata.n_vars,
}
print(json.dumps(qc_trace, indent=2))

### 5c — Scrublet per sample (compute → rescue failures → filter)

Doublet detection is run **per sample**, not on the concatenated object, because doublet rate is a per-library property.

**Why this cell is a two-pass rewrite (2026-06-04).** The first run exposed a silent failure: 14 of 56 samples returned a doublet rate of 0.0–0.5% while the other 42 sat at 5.6–19%, with a hard gap and nothing in between. A real 10x doublet distribution is continuous and never ca. 0% (random co-encapsulation alone gives ≥2%) — that bimodal gap is the signature of Scrublet's **automatic threshold detection failing** on those 14 samples (it could not separate the simulated-doublet mode), not of clean libraries. Calling `scrub_doublets(verbose=False)` also suppressed the warning Scrublet prints when this happens.

The fix decouples the three jobs the old single-shot cell fused:

1. **Compute** doublet scores + Scrublet's own (auto) calls for every sample — no filtering yet.
2. **Rescue** samples whose auto-rate is below `AUTO_FAIL_RATE` (2%): assume they share the cohort's true rate (median of the samples where detection worked) and re-call the top fraction by `doublet_score` within each. Uniform and reproducible — no per-sample manual histogram thresholds, which a committed notebook couldn't reproduce.
3. **Filter once** at the end.

A reporting cross-tab (failed samples × diagnosis × `apoe_carrier`) is printed so the `_OUTPUT` writeup can judge whether the residual pattern is mere hygiene (failures scatter) or a confound with the biological axis (failures cluster in a stratum). **Read that table before trusting eval #2.**

**Memory.** The first run sat at the edge of the high-RAM runtime. This cell forces float32 CSR up front (halves every downstream copy), slices each sample sparsely with explicit `del` + `gc` per iteration, does a single filter copy at the end with `gc` immediately after, and prints `[RAM]` probes at each stage so the true peak/headroom is observable rather than guessed.

In [ ]:
import numpy as np
import gc
import pandas as pd
import scrublet as scr
from scipy.sparse import csr_matrix, issparse

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:18s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB used ({m.percent:.0f}%)")
except Exception:
    def _ram(tag):
        print(f"[RAM] {tag}: psutil unavailable")

# --- Memory guard: float32 CSR before anything ----------------------------------
# X is raw counts here (no normalization yet — scVI handles that later); Scrublet
# wants raw counts. Forcing float32 CSR halves the footprint of every per-sample
# slice AND the final filter copy (the spike). Guarded so it is a no-op spending no
# transient memory when X is already float32 CSR.
_ram("5c start")
if not (issparse(adata.X) and adata.X.getformat() == "csr"):
    adata.X = csr_matrix(adata.X)
if adata.X.dtype != np.float32:
    print(f"Casting X {adata.X.dtype} -> float32 (one-time transient)")
    adata.X = adata.X.astype(np.float32)
    gc.collect()

candidate_sample_cols = ["sample_id", "library_id", "batch", "sample", "orig.ident"]
SAMPLE_COL = next((c for c in candidate_sample_cols if c in adata.obs.columns), None)
if SAMPLE_COL is None:
    print(f"WARNING: no sample column found (checked {candidate_sample_cols}). "
          "Running Scrublet on full object as single batch — doublet rate may be inflated.")
    SAMPLE_COL = "_batch"
    adata.obs[SAMPLE_COL] = "all"
EXPECTED_DOUBLET_RATE = 0.10   # Scrublet prior
AUTO_FAIL_RATE        = 0.02   # auto-call below this = threshold-detection failure, not a clean sample
print(f"Scrublet per sample using column: {SAMPLE_COL}")

# --- Pass 1: compute scores per sample, DO NOT filter yet ------------------------
# Splitting compute from filter is the whole point of this rewrite: Scrublet returns
# ~0% on samples where it cannot locate the simulated-doublet mode (auto-threshold
# failure). Computing first lets us catch those and re-call them below BEFORE any
# cell is dropped — instead of silently keeping their doublets (the bug in the
# first run: 14/56 samples read 0.0-0.5% with a hard gap to 5.6%, impossible for a
# real 10x doublet distribution).
adata.obs["doublet_score"]      = np.nan
adata.obs["scrublet_auto_call"] = False

sample_auto_rate = {}
for sample in adata.obs[SAMPLE_COL].unique():
    mask  = (adata.obs[SAMPLE_COL] == sample).values
    sub_X = adata.X[mask]                      # small sparse slice, freed each iter
    if sub_X.shape[0] < 30:
        print(f"  {sample}: n={sub_X.shape[0]} — too small, skipping")
        del sub_X
        continue
    sclt = scr.Scrublet(sub_X, expected_doublet_rate=EXPECTED_DOUBLET_RATE)
    scores, calls = sclt.scrub_doublets(verbose=False)
    if calls is None:                          # hard failure -> treat as 0%, rescued below
        calls = np.zeros(sub_X.shape[0], dtype=bool)
    rate = float(np.mean(calls))
    sample_auto_rate[sample] = rate
    adata.obs.loc[mask, "doublet_score"]      = scores
    adata.obs.loc[mask, "scrublet_auto_call"] = calls
    print(f"  {sample}: n={sub_X.shape[0]:5d}  auto_doublets={int(calls.sum()):5d} ({100*rate:4.1f}%)")
    del sclt, scores, calls, sub_X, mask
    gc.collect()
_ram("after compute loop")

# --- Pass 2: rescue samples where auto-threshold failed --------------------------
# A true 10x doublet rate is never ~0% (random co-encapsulation alone gives >=2%);
# a sample below AUTO_FAIL_RATE means Scrublet could not separate the doublet mode,
# not that the sample is clean. Fallback: assume failed samples share the cohort's
# true rate (median of samples where detection worked) and call that top fraction by
# doublet_score within each. Uniform + reproducible — no per-sample manual thresholds
# to eyeball (which a committed notebook could not reproduce anyway).
failed = sorted(s for s, r in sample_auto_rate.items() if r <  AUTO_FAIL_RATE)
ok     = sorted(s for s, r in sample_auto_rate.items() if r >= AUTO_FAIL_RATE)
cohort_median_rate = float(np.median([sample_auto_rate[s] for s in ok])) if ok else EXPECTED_DOUBLET_RATE
print(f"\nauto-threshold OK: {len(ok)} samples | FAILED (<{AUTO_FAIL_RATE:.0%}): {len(failed)} samples")
print(f"cohort median doublet rate (OK samples): {cohort_median_rate:.1%}")
print(f"failed samples re-called at cohort rate: {failed}")

adata.obs["predicted_doublet"] = adata.obs["scrublet_auto_call"].astype(bool)
for sample in failed:
    mask = (adata.obs[SAMPLE_COL] == sample).values
    s    = adata.obs.loc[mask, "doublet_score"].to_numpy()
    if np.all(np.isnan(s)):
        continue
    k         = max(int(np.ceil(cohort_median_rate * len(s))), 0)
    thr       = np.sort(s)[::-1][k - 1] if k > 0 else np.inf
    new_calls = s >= thr
    adata.obs.loc[mask, "predicted_doublet"] = new_calls
    print(f"  {sample}: fallback thr={thr:.3f} -> {int(new_calls.sum())} doublets ({100*new_calls.mean():.1f}%)")

# --- Reporting: does the auto-failure pattern correlate with the biology? ---------
# If failed samples scatter across diagnosis x APOE, residual-doublet risk is hygiene.
# If they cluster in a stratum, uneven doublet removal is confounded with the
# biological axis going into integration. This table is what decides which — read it
# in the _OUTPUT writeup before trusting eval #2.
if {"diagnosis", "apoe_carrier"}.issubset(adata.obs.columns):
    donor_meta = (adata.obs[[SAMPLE_COL, "diagnosis", "apoe_carrier"]]
                  .drop_duplicates(SAMPLE_COL).set_index(SAMPLE_COL))
    donor_meta["scrublet_failed"] = donor_meta.index.isin(failed)
    print("\n=== Scrublet auto-threshold failure vs diagnosis x apoe_carrier ===")
    print(pd.crosstab([donor_meta["diagnosis"], donor_meta["apoe_carrier"]],
                      donor_meta["scrublet_failed"], margins=True))
else:
    print("\n(diagnosis/apoe_carrier not on obs — skipping failure-vs-biology cross-tab)")

# --- Single filter (the RAM spike) -----------------------------------------------
# Boolean-subset + copy transiently holds old X and new X together (~1.9x X). gc
# immediately after drops the old matrix; the _ram brackets make the real peak
# observable so we know the true headroom on this runtime.
_ram("before filter")
n_before = adata.n_obs
keep = ~adata.obs["predicted_doublet"].to_numpy()
adata = adata[keep].copy()
gc.collect()
_ram("after filter")

qc_trace["n_cells_after_scrublet"]   = int(adata.n_obs)
qc_trace["scrublet_removed"]         = int(n_before - adata.n_obs)
qc_trace["scrublet_failed_samples"]  = failed
qc_trace["scrublet_n_failed"]        = len(failed)
qc_trace["scrublet_cohort_med_rate"] = round(cohort_median_rate, 4)
qc_trace["sample_col"]               = SAMPLE_COL
print(f"\nAfter Scrublet (auto + fallback): {adata.n_obs} cells "
      f"({qc_trace['scrublet_removed']} removed; {len(failed)} samples rescued)")


### 5d — Mito threshold sensitivity (3% / 5% / 10%)

Sanity check on the locked 5% default. Runs on `pre_mito_obs` (snapshotted in 5b, post count/gene filter, pre mito filter) — the sweep needs the high-mito cells the 5b filter removed, so it cannot run after filtering and cannot be deferred to the integration notebook (by then each study is already mito-filtered and the high-mito cells are gone).

The sharp question is not "how many cells survive" but "does the mito threshold preferentially cut the niche?" So the sweep reports, at each threshold, both total cells retained and astrocyte / microglia cells retained.

Two caveats, both honest limitations of doing this at the per-study stage:

- **Niche labels are author-shipped (a proxy).** The project's harmonized astro/microglia labels only exist post-integration (ITS design). Li 2025's own annotation — if present on GEO — is used here as a stand-in for the diagnostic. If no cell-type column is found, the sweep falls back to total cells only and says so.
- **Snapshot is pre-Scrublet.** 5c (doublet removal) runs after 5b, so `pre_mito_obs` still contains doublets. Acceptable for a relative-sensitivity check across thresholds.

If astro/microglia counts barely move across 3% / 5% / 10%, the 5% choice is not load-bearing. If they swing substantially, the threshold needs justification (recorded for the `_WITH_OUTPUT` writeup, not concluded here).

In [ ]:
# Sweep runs on pre_mito_obs (post count/gene filter, pre mito filter) from 5b.
MITO_THRESHOLDS = [3.0, 5.0, 10.0]

# Find an author-shipped cell-type label (proxy for the project's post-integration
# harmonized labels — see 5d markdown). Fall back to totals-only if none is found.
candidate_celltype_cols = [
    "cell_type", "celltype", "CellType", "cell_type_label",
    "subclass", "subclass_label", "major_celltype", "annotation", "cluster_name",
]
CELLTYPE_COL = next((c for c in candidate_celltype_cols if c in pre_mito_obs.columns), None)

if CELLTYPE_COL:
    _ct = pre_mito_obs[CELLTYPE_COL].astype(str).str.lower()
    astro_mask = _ct.str.contains("astro")
    micro_mask = _ct.str.contains("micro")
    print(f"Cell-type column: {CELLTYPE_COL!r}")
    print(f"Unique values: {sorted(pre_mito_obs[CELLTYPE_COL].astype(str).unique())}")
    print(f"Matched  astro={int(astro_mask.sum())}  micro={int(micro_mask.sum())}")
    if astro_mask.sum() == 0 or micro_mask.sum() == 0:
        print("WARNING: astro or micro matched 0 cells — labels may be abbreviated "
              "(e.g. 'AST' / 'MG'). Check the unique values above and adjust if needed.")
else:
    astro_mask = micro_mask = None
    print("No cell-type column found — sweep reports total cells only.")
    print(f"Columns available: {list(pre_mito_obs.columns)}")

n_pre = len(pre_mito_obs)
sweep = {}
for t in MITO_THRESHOLDS:
    keep = pre_mito_obs["pct_counts_mt"] < t
    row = {"total_retained": int(keep.sum()), "total_frac": round(float(keep.mean()), 4)}
    if CELLTYPE_COL:
        row["astro_retained"] = int((keep & astro_mask).sum())
        row["micro_retained"] = int((keep & micro_mask).sum())
    sweep[f"mito_lt_{t}"] = row

qc_trace["mito_sensitivity"] = {
    "status":            "computed",
    "pre_mito_pool":     n_pre,
    "locked_threshold":  QC_PARAMS["max_mito_pct"],
    "celltype_col":      CELLTYPE_COL,
    "celltype_source":   "author-shipped labels (proxy)" if CELLTYPE_COL else "unavailable",
    "astro_total":       (int(astro_mask.sum()) or None) if CELLTYPE_COL else None,
    "micro_total":       (int(micro_mask.sum()) or None) if CELLTYPE_COL else None,
    "snapshot":          "post count/gene filter, pre mito filter, pre Scrublet",
    "thresholds":        sweep,
}
print("\n" + json.dumps(qc_trace["mito_sensitivity"], indent=2))

## 6 — Audit A (partial): niche-critical gene presence

Full Audit A is FM-vocab intersection (Geneformer Ensembl + scGPT HGNC) — that needs the FM envs (Geneformer on native Python, see `requirements_geneformer.txt`; scGPT on Python 3.10, see `requirements_scgpt.txt`) and lives in later notebooks.

This partial Audit A asks one question: are the niche-critical genes present in Li 2025's `var_names` (or `gene_symbols`) at all? If any are missing here, they will definitely be missing downstream — fail early.

Niche-critical (locked 2026-05-22): `APOE`, `TREM2`, `MS4A6A`, `CLU`, `GFAP`, `AQP4`, `AIF1` (IBA1), `CSF1R`.

### 6a — Check presence and write Audit A (Li 2025 row) to audit_report.json

Searches `var_names` and the `gene_symbols` column (if Ensembl-named). Reports per-gene status. Final Audit A pass-criterion (zero dropped, across all studies × both FMs) is only computable after all three studies + FM vocab intersection — this writes only the Li 2025 row.

In [ ]:
from src.data_loaders import NICHE_CRITICAL_GENES

if looks_ensembl and SYMBOL_COL and SYMBOL_COL in adata.var.columns:
    gene_pool = set(adata.var[SYMBOL_COL].astype(str))
else:
    gene_pool = set(adata.var_names.astype(str))

per_gene = {g: (g in gene_pool) for g in NICHE_CRITICAL_GENES}
missing  = [g for g, present in per_gene.items() if not present]

audit_a_li2025 = {
    "study":           "Li 2025",
    "vocab_format":    "Ensembl" if looks_ensembl else "HGNC",
    "symbol_col_used": SYMBOL_COL if looks_ensembl else None,
    "n_genes_post_qc": adata.n_vars,
    "per_gene":        per_gene,
    "missing":         missing,
    "status":          "pass" if not missing else "fail",
    "note":            "partial — FM-vocab intersection deferred to FM env notebook",
}

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
if os.path.exists(AUDIT_REPORT_PATH):
    with open(AUDIT_REPORT_PATH) as f:
        report = json.load(f)
else:
    report = {}
audit_a = report.get("audit_a_vocab_niche_critical", {"per_study": {}})
audit_a["per_study"]["Li 2025"] = audit_a_li2025
audit_a["status"] = "partial"
report["audit_a_vocab_niche_critical"] = audit_a
report["li2025_qc_trace"]              = qc_trace
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(audit_a_li2025, indent=2))

## 7 — Save processed AnnData and clean up raw

Processed h5ad goes to Drive (`*.h5ad` is gitignored — it lives outside the repo). Raw GEO download is deleted per Audit F (`keep=False`) — saves several GB.

### 7a — Save Li 2025 processed AnnData

Sparse CSR enforced before write — memory discipline from the integration-env compute constraints (Colab high-RAM ca. 50 GB, concatenated multi-study tight).

In [ ]:
from scipy.sparse import csr_matrix, issparse

if not issparse(adata.X) or adata.X.getformat() != "csr":
    adata.X = csr_matrix(adata.X)

PROCESSED_DIR  = os.path.join(DRIVE_ROOT, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)
PROCESSED_PATH = os.path.join(PROCESSED_DIR, "li2025_processed.h5ad")
adata.write_h5ad(PROCESSED_PATH, compression="gzip")

size_mb = os.path.getsize(PROCESSED_PATH) / 1024**2
print(f"Wrote {PROCESSED_PATH}  ({size_mb:.1f} MB)")

### 7b — Delete raw download

Per Audit F: raw downloads are `keep=False`, deleted after per-study processed AnnData saved. Saves the GEO supplementary footprint (several GB) back.

In [ ]:
import shutil

raw_size_gb = sum(os.path.getsize(os.path.join(dp, f))
                  for dp, _, fs in os.walk(RAW_DIR) for f in fs) / 1024**3
print(f"Raw dir size before delete: {raw_size_gb:.2f} GB")

shutil.rmtree(RAW_DIR)
print(f"Deleted {RAW_DIR}")

## 8 — Handoff to colab_02

Prints audit summary + git commit instructions. If Li 2025 is clean here, the same pattern scales to SEA-AD (colab_02) and Haney 2024 (colab_03).

### 8a — Summary + commit instructions

Lists artifacts written and the explicit `git add / commit / push` commands. Processed h5ad is gitignored (lives only on Drive); freeze, env JSON, and audit_report are committed.

In [ ]:
import shlex

with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

print("=== Audit summary ===")
for k, v in report.items():
    print(f"  {k}: {v.get('status') if isinstance(v, dict) else v}")

print("\n=== Artifacts ===")
print(f"  committable:  {FREEZE_PATH}")
print(f"  committable:  {ENV_JSON_PATH}")
print(f"  committable:  {AUDIT_REPORT_PATH}")
print(f"  drive-only:   {PROCESSED_PATH}")

rel_paths = [os.path.relpath(p, REPO_PATH) for p in [FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH]]
msg = f"colab_01: Li 2025 QC + Audit A (partial) ({TODAY})"
print("\n=== Commit + push ===")
for c in [
    f"cd {REPO_PATH} && git add {' '.join(rel_paths)}",
    f"cd {REPO_PATH} && git commit -m {shlex.quote(msg)}",
    f"cd {REPO_PATH} && git push",
]:
    print(f"  {c}")

print("\nNext: colab_02_seaad_qc.ipynb (SEA-AD download + secondary_qc + Audit A row).")